# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Access metadata as an object
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List all record sets with their @id and their fields
record_sets = list(dataset.record_sets)
print(f"Available Record Sets (@id):\n---------------------------")
for rs in record_sets:
    print(f"- {rs.id} | {rs.name}")
    print(f"  Fields:")
    for field in rs.fields:
        print(f"    - {field.id} | {field.name}")
    print()

# Show first record from each set as example
for rs in record_sets:
    print(f"First record of {rs.id}:\n---------------------------")
    try:
        recs = dataset.records(record_set=rs.id)
        # Print just one record as example
        print(next(recs))
    except Exception as e:
        print(f"  Could not load: {e}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from all record sets into DataFrames, referenced by @id
dataframes = {}

print("Loading all record sets into pandas DataFrames...\n")
for rs in record_sets:
    try:
        records = list(dataset.records(record_set=rs.id))
        df = pd.DataFrame(records)
        dataframes[rs.id] = df
        print(f"Loaded: {rs.id} | {rs.name} -> shape: {df.shape}")
    except Exception as e:
        print(f"Could not load records from {rs.id}: {e}")

# Choose the main quantitative record set for further examples (by inspecting above)
main_rs_id = None
for rs in record_sets:
    # Heuristic: look for record set mentioning proteins
    if 'protein' in rs.id.lower() or 'protein' in rs.name.lower():
        main_rs_id = rs.id
        break
if main_rs_id is None:
    # Fallback: pick the first non-empty DataFrame
    for rs in record_sets:
        if dataframes.get(rs.id, pd.DataFrame()).shape[1] > 2:
            main_rs_id = rs.id
            break

print("\nMain record set for EDA: ", main_rs_id)
print("Columns in main record set:")
print(dataframes[main_rs_id].columns.tolist())
dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Pick most interesting numeric field for analysis by inspecting the columns
df = dataframes[main_rs_id]
numeric_candidate = None

# Try common mass-spectrometry column names (abundance, MW, etc.):
for col in df.columns:
    if col.lower() in ['abundance', 'normalized_abundance', 'coverage', 'peptides', 'unique_peptides', 'mw', 'mass', 'intensity']:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_candidate = col
            break
if numeric_candidate is None:
    # Fallback: any numeric column
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_candidate = col
            break

print(f"Numeric field selected: {numeric_candidate}")
# Set a numeric threshold -- e.g., median
if numeric_candidate is not None:
    median_thresh = df[numeric_candidate].median()
    filtered_df = df[df[numeric_candidate] > median_thresh]
    print(f"Filtered records with {numeric_candidate} > {median_thresh} (median):\n")
    print(filtered_df.head())

    # Normalize this field
    filtered_df[f"{numeric_candidate}_normalized"] = (filtered_df[numeric_candidate] - filtered_df[numeric_candidate].mean()) / filtered_df[numeric_candidate].std()
    print(f"\nNormalized {numeric_candidate} for filtered records:")
    print(filtered_df[[numeric_candidate, f"{numeric_candidate}_normalized"]].head())

    # Try grouping by a categorical field (e.g., 'description', 'accession')
    group_field = None
    for cat in ['description', 'accession', 'name']:
        if cat in df.columns:
            group_field = cat
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_candidate].mean()
        print(f"\nGrouped data by {group_field} (mean of {numeric_candidate}):")
        print(grouped_df.head())
else:
    print("No numeric field found for analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_candidate is not None:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_candidate].dropna(), bins=40, kde=True)
    plt.title(f"Distribution of {numeric_candidate}")
    plt.xlabel(numeric_candidate)
    plt.ylabel('Count')
    plt.show()
    # If a group field is available, show boxplot
    if group_field:
        plt.figure(figsize=(12, 5))
        sns.boxplot(x=group_field, y=numeric_candidate, data=filtered_df)
        plt.title(f"{numeric_candidate} by {group_field} (Filtered)")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.


- Successfully loaded the metadata and main protein records table using the Croissant schema.
- Explored available record sets and identified main quantitative fields such as protein abundance or molecular weight.
- Performed basic filtering, normalization, and grouping operations, demonstrating typical data preparation tasks.
- Visualized the main quantitative field's distribution and group-wise variation.

This workflow demonstrates reproducible access and FAIR data handling using the `mlcroissant` library.
<br><br>For further biological or statistical analysis, refer to the field `@id`s used in cell outputs and refer back to the Croissant schema URL for the latest documentation.